# 05 — Interview Question Bank: Model Answers with Runnable Checks

A curated bank of **63 interview questions** with concise, interview-ready model
answers and one deeper follow-up each, organised into nine sections that mirror a
typical data-engineering loop:

**Python · SQL · Spark and PySpark · Data modelling and warehousing ·
Lakehouse and file formats · Pipelines and orchestration · Streaming ·
Cloud, cost and reliability · Behavioural and system design**

Fifteen of the questions are paired with a **runnable "verify your answer" cell**
that proves the behaviour in standard-library Python (including `sqlite3`). Nothing
here needs pandas, PySpark or a cluster.

## How to use this notebook

1. **Read the question, then answer out loud before scrolling.** The value is in
   retrieval practice, not in reading the model answer.
2. Compare against **A.** — it is deliberately 3–6 sentences, the length of a good
   interview answer. If you drifted, note what you missed rather than re-reading.
3. Treat **Follow-up:** as the real interview. Interviewers rarely stop at the first
   answer; rehearse the follow-up too.
4. Where a section ends with **"Verify your answer"**, run the cell. Every check
   uses only the standard library, contains `assert`s, and prints a short pass
   message. A failing assert means your mental model of the mechanism is wrong —
   fix that before moving on.
5. Restart-and-run-all occasionally (`Kernel → Restart & Run All`) so you know the
   notebook is reproducible end to end.

> The checks are illustrative models, not production code: they reduce a mechanism
> (a shuffle, a watermark, a lakehouse snapshot) to the smallest thing that is
> still faithful enough to reason about.

## Python

Core Python questions that appear in almost every data-engineering screen. The runnable checks below verify the behaviours that are easiest to get wrong out loud.

**Q.** What is wrong with a mutable default argument, and how do you fix it?

**A.** Default values are evaluated once, when the `def` statement executes, so a list or dict default is shared by every call that omits the argument. `def add(item, acc=[])` therefore accumulates state across calls, which is a classic production bug that is invisible in tests that call the function once. Fix it by defaulting to `None` and creating the container inside the body, or by using an immutable default and an explicit copy. Pass the container explicitly when the caller wants to control it.

**Follow-up:** Does the same trap apply to class attributes and function attributes, and how do you decide when a shared object is intentional?

**Q.** When would you use a generator instead of building a list?

**A.** Generators produce items lazily, so memory stays roughly constant regardless of input size, which matters when you stream a multi-GB CSV or a Kafka backlog through a transformation. They compose cleanly as a pipeline of `yield` stages and short-circuit naturally with `islice` or `next`. You give up random access, `len()`, and re-iteration, because a generator is single-use. Reach for a list when the data is small or you need to traverse it repeatedly.

**Follow-up:** How do generator expressions differ from list comprehensions in evaluation time, and what happens if you use `return` inside a generator?

**Q.** Why do closures created in a loop all see the final value of the loop variable?

**A.** Python closures capture variables by reference, not by value, so every lambda created in the loop shares one cell and sees `i`'s final value after the loop finishes — this is late binding. The fixes are a default argument (`lambda i=i: i`), `functools.partial`, or a factory function that creates a fresh scope per iteration. The same pattern bites in callback registration and in task factories for Airflow or Dagster, so bind the value explicitly at creation time.

**Follow-up:** What does `nonlocal` do, and how does it differ from `global`?

**Q.** What does `functools.wraps` do, and why does it matter in decorators?

**A.** A decorator returns a wrapper function, which by default carries the wrapper's `__name__`, docstring and signature instead of the wrapped function's, breaking logging, introspection and any framework that dispatches on names. `functools.wraps(f)` copies `__name__`, `__doc__`, `__module__`, `__qualname__` and `__wrapped__` from `f` onto the wrapper. Use it in every decorator; when arguments and signatures must be preserved for callers, pair it with `inspect.signature` if the wrapper *args/**kwargs shape matters.

**Follow-up:** How would you write a decorator that takes its own arguments, such as `@retry(times=3)`?

**Q.** How do context managers work, and when is one better than `try/finally`?

**A.** A context manager implements `__enter__` and `__exit__`; the `with` block guarantees `__exit__` runs even if the body raises, and it can suppress or translate exceptions. `@contextlib.contextmanager` lets you write one from a generator with a single `yield`, which is usually shorter than a class and equally safe. Use them for anything that must be paired — files, transactions, DB connections, timers, temporary Spark config. `try/finally` still works, but `with` makes the pairing part of the contract.

**Follow-up:** How would you make a context manager that is also a decorator, and what does `contextlib.ExitStack` solve?

**Q.** Explain shallow vs deep copy, and what makes an object hashable.

**A.** `copy.copy` duplicates the outer container but shares nested objects, so mutating a nested list is visible through both copies; `copy.deepcopy` copies recursively, at the cost of speed and possible failure on un-copyable resources. An object is hashable when its `__hash__` is stable for its lifetime and consistent with `__eq__`; mutable containers are unhashable by design. Defining `__eq__` without `__hash__` sets `__hash__` to `None`, which silently makes instances unusable as dict keys. If you need mutability and a key, store an immutable derived value.

**Follow-up:** Why can two objects be equal but have different hashes, and what breaks in a dict if a key is mutated after insertion?

**Q.** What is the GIL, and how should it change your concurrency choice?

**A.** The GIL lets only one thread execute Python bytecode at a time, so threads give no CPU parallelism for pure-Python work, but they do help I/O-bound work because the GIL is released during I/O. For CPU-bound work use `multiprocessing` or `concurrent.futures.ProcessPoolExecutor` and accept serialization overhead. `asyncio` fits many concurrent network calls with far less overhead than threads. In data engineering this is mostly a reason to push heavy compute into Spark or the warehouse and keep Python as the coordinator.

**Follow-up:** How do C extensions such as NumPy or orjson bypass the GIL, and what changes with free-threaded Python 3.13?

### Verify your answer: mutable default arguments

In [1]:
def collect_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

assert collect_bad("a") == ["a"]
assert collect_bad("b") == ["a", "b"]          # the shared-default bug

def collect_ok(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

assert collect_ok("a") == ["a"]
assert collect_ok("b") == ["b"]                # fresh list every call
assert collect_ok("c", ["seed"]) == ["seed", "c"]
print("mutable default check passed")

mutable default check passed


### Verify your answer: late-binding closures

In [2]:
funcs = [lambda: i for i in range(3)]
assert [f() for f in funcs] == [2, 2, 2]       # late binding: all share one cell

funcs_fixed = [lambda i=i: i for i in range(3)]
assert [f() for f in funcs_fixed] == [0, 1, 2]

from functools import partial

def keep(i):
    return i

funcs_partial = [partial(keep, i) for i in range(3)]
assert [f() for f in funcs_partial] == [0, 1, 2]
print("closure late-binding check passed")

closure late-binding check passed


### Verify your answer: generators and order-preserving dedupe

In [3]:
import itertools

def dedupe(iterable):
    seen = set()
    for item in iterable:
        if item not in seen:
            seen.add(item)
            yield item

data = ["a", "b", "a", "c", "b", "d"]
assert list(dedupe(data)) == ["a", "b", "c", "d"]

def take(n, iterable):
    return list(itertools.islice(iterable, n))

def naturals():
    n = 0
    while True:
        yield n
        n += 1

assert take(5, naturals()) == [0, 1, 2, 3, 4]           # lazy: safe on an infinite source
assert take(3, dedupe([1, 1, 2, 2, 3, 3, 4])) == [1, 2, 3]
print("generator/dedupe check passed")

generator/dedupe check passed


### Verify your answer: dunder methods on a value class

In [4]:
class Money:
    def __init__(self, cents):
        self.cents = int(cents)
    def __add__(self, other):
        return Money(self.cents + other.cents)
    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents
    def __hash__(self):
        return hash(self.cents)
    def __repr__(self):
        return "Money(%d)" % self.cents

a, b = Money(150), Money(250)
assert a + b == Money(400)
assert repr(a + b) == "Money(400)"
assert a != b and not (a == 42)
assert len({a, Money(150), b}) == 2            # __hash__ and __eq__ agree
assert {Money(150): "price"}[a] == "price"
print("dunder class check passed")

dunder class check passed


### Verify your answer: context managers and cleanup on error

In [5]:
import contextlib

class Resource:
    def __init__(self):
        self.open = False
        self.closed = False
    def __enter__(self):
        self.open = True
        return self
    def __exit__(self, exc_type, exc, tb):
        self.open = False
        self.closed = True
        return False                           # never swallow exceptions

r = Resource()
with r as handle:
    assert handle is r and r.open is True
assert r.closed is True and r.open is False

r2 = Resource()
try:
    with r2:
        raise ValueError("boom")
except ValueError:
    pass
assert r2.closed is True                       # __exit__ ran despite the error

events = []

@contextlib.contextmanager
def session(name):
    events.append(("open", name))
    try:
        yield name.upper()
    finally:
        events.append(("close", name))

with session("tmp") as s:
    assert s == "TMP"
assert events == [("open", "tmp"), ("close", "tmp")]
print("context manager check passed")

context manager check passed


## SQL

Window functions, joins and NULL semantics dominate SQL interviews. The runnable checks use an in-memory SQLite database so you can prove the answers.

**Q.** Explain ROW_NUMBER, RANK and DENSE_RANK, and how you use them for top-N per group.

**A.** All three are window functions: `ROW_NUMBER` gives unique consecutive numbers, `RANK` leaves gaps after ties, and `DENSE_RANK` does not. To get top-N per group you must wrap the window in a subquery or CTE and filter on the rank, because a window function cannot appear in `WHERE`. Use `ROW_NUMBER` when you need exactly one row per key to deduplicate, and `RANK` or `DENSE_RANK` when ties should all be kept. Always add a deterministic tie-breaker to `ORDER BY`, or results vary between runs.

**Follow-up:** Which one would you use to deduplicate an append-only CDC table, and why does a non-deterministic ordering matter there?

**Q.** Write the second-highest salary query and handle ties and NULLs.

**A.** `SELECT MAX(salary) FROM employee WHERE salary < (SELECT MAX(salary) FROM employee)` returns the value, works with duplicate top salaries, and returns NULL on an empty or single-row table. A window version, `DENSE_RANK() OVER (ORDER BY salary DESC) = 2`, returns the employees who actually earn it. `ORDER BY salary DESC LIMIT 1 OFFSET 1` is the shortest but misbehaves with ties at the top. Clarify whether the interviewer wants the value or the people, then filter NULLs explicitly.

**Follow-up:** How would you return the Nth highest salary for an arbitrary N, and which approach scales best on a large table?

**Q.** What are the NULL pitfalls in joins?

**A.** `NULL = NULL` evaluates to unknown, so joining on a nullable key silently drops rows unless you normalize with `COALESCE` or a surrogate key. `NOT IN (SELECT ...)` returns zero rows if the subquery contains a single NULL, because the comparison is unknown for every row; the reliable anti-join is `LEFT JOIN ... WHERE right.key IS NULL` or `NOT EXISTS`. Outer joins also fan out when the right side is not unique, inflating downstream aggregates. Compare row counts before and after every join and enforce non-null keys by contract.

**Follow-up:** How do `NOT IN`, `NOT EXISTS` and `LEFT JOIN ... IS NULL` differ in performance and semantics on a large warehouse table?

**Q.** WHERE vs HAVING, and GROUP BY vs DISTINCT?

**A.** `WHERE` filters rows before aggregation and can enable partition pruning; `HAVING` filters after aggregation and is the only way to filter on an aggregate. `GROUP BY` and `DISTINCT` can produce identical results, but `GROUP BY` supports aggregates and generally optimizes better, while `DISTINCT` is a set operation over the projected columns. Do not use `SELECT DISTINCT` to paper over a join fan-out — fix the grain instead. Push filters as early as possible so less data moves through the query.

**Follow-up:** What is the logical execution order of a SQL query, and where do window functions fit into it?

**Q.** Correlated subquery vs join — when is each appropriate?

**A.** A correlated subquery conceptually runs once per outer row, so it can be expensive, but most optimizers decorrelate it into a join and it stays very readable for `EXISTS`/`NOT EXISTS` presence checks. Joins are the right tool when you need columns from both sides or an aggregation, and a join plus `GROUP BY` replaces most correlated scalar subqueries. Prefer `EXISTS` over `IN` when the subquery is large or nullable. As always, compare the plans rather than assuming.

**Follow-up:** How would you rewrite a correlated subquery that computes a per-row maximum into a window function?

**Q.** How do you write an incremental upsert and dedupe in SQL?

**A.** Use `MERGE INTO target USING staging ON key WHEN MATCHED THEN UPDATE WHEN NOT MATCHED THEN INSERT`, which is atomic and idempotent as long as the source is deduplicated first. Deduplicate staging with `ROW_NUMBER() OVER (PARTITION BY key ORDER BY updated_at DESC) = 1` so one statement never updates the same key twice. Restrict the source to the changed window with a watermark such as `updated_at > last_run` to keep the merge incremental. On engines without `MERGE`, use delete-then-insert or an insert-overwrite of the affected partitions.

**Follow-up:** What are the risks of MERGE on a large table — retries, concurrent writers, upstream deletes — and how do you mitigate them?

**Q.** How do you diagnose and fix a slow SQL query?

**A.** Start with the plan (`EXPLAIN` or `EXPLAIN ANALYZE`) and look for full scans, exploding joins and exchange/shuffle operators, then check whether the estimated row counts are wildly off. Verify that predicates are sargable and that partition or clustering columns are being pruned, and refresh statistics if estimates are stale. Fix the model if joins fan out: pre-aggregate, filter earlier, or change the grain. Watch for skew in join keys, and for functions or casts applied to a filtered column, which disable index and partition pruning.

**Follow-up:** What is predicate pushdown, and why does wrapping a filtered column in a function or cast defeat pruning?

### Verify your answer: second-highest salary in sqlite3

In [6]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE employee (id INTEGER PRIMARY KEY, name TEXT, salary INTEGER)")
con.executemany(
    "INSERT INTO employee VALUES (?, ?, ?)",
    [(1, "ana", 90), (2, "bo", 120), (3, "cy", 120), (4, "di", 75)],
)

# Approach 1: max below the max. Handles duplicate top salaries; NULL when none exists.
q1 = "SELECT MAX(salary) FROM employee WHERE salary < (SELECT MAX(salary) FROM employee)"
assert con.execute(q1).fetchone()[0] == 90

# Approach 2: window function, and it also tells you who earns it.
q2 = """
SELECT name, salary FROM (
  SELECT name, salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rk
  FROM employee
) WHERE rk = 2 ORDER BY name
"""
assert con.execute(q2).fetchall() == [("ana", 90)]

# Edge case: no second value -> NULL, not an error.
con.execute("CREATE TABLE only_one (salary INTEGER)")
con.execute("INSERT INTO only_one VALUES (100)")
assert con.execute(q1.replace("employee", "only_one")).fetchone()[0] is None
con.close()
print("sqlite second-highest-salary check passed")

sqlite second-highest-salary check passed


### Verify your answer: top-N per group with window functions

In [7]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE sales (region TEXT, rep TEXT, amount INTEGER)")
con.executemany(
    "INSERT INTO sales VALUES (?, ?, ?)",
    [
        ("east", "ana", 100), ("east", "bo", 80), ("east", "cy", 60),
        ("west", "di", 90), ("west", "ed", 70), ("west", "fi", 50),
    ],
)

q = """
SELECT region, rep, amount FROM (
  SELECT region, rep, amount,
         ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC, rep) AS rn
  FROM sales
) WHERE rn <= 2 ORDER BY region, amount DESC
"""
assert con.execute(q).fetchall() == [
    ("east", "ana", 100), ("east", "bo", 80),
    ("west", "di", 90), ("west", "ed", 70),
]

# Ties: 100, 90, 90, 50 -> which ranking function returns how many rows for top-2?
tied = ("SELECT 'a' AS rep, 100 AS amount "
        "UNION ALL SELECT 'b', 90 "
        "UNION ALL SELECT 'c', 90 "
        "UNION ALL SELECT 'd', 50")

def top2_count(function_name):
    query = (
        "SELECT COUNT(*) FROM (SELECT %s() OVER (ORDER BY amount DESC) AS rk "
        "FROM (%s)) WHERE rk <= 2" % (function_name, tied)
    )
    return con.execute(query).fetchone()[0]

assert top2_count("ROW_NUMBER") == 2     # 1, 2 -> exactly two rows
assert top2_count("RANK") == 3           # 1, 2, 2 -> both tied 90s are kept
assert top2_count("DENSE_RANK") == 3     # 1, 2, 2 -> no gap, so same three rows
con.close()
print("sqlite top-N-per-group check passed")

sqlite top-N-per-group check passed


### Verify your answer: NULL-safe anti-joins

In [8]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE orders (id INTEGER, customer_id INTEGER)")
con.execute("CREATE TABLE customers (id INTEGER, name TEXT)")
con.executemany("INSERT INTO orders VALUES (?, ?)", [(1, 10), (2, 20), (3, None)])
# One customer row has a NULL id, which is what makes NOT IN dangerous.
con.executemany("INSERT INTO customers VALUES (?, ?)", [(10, "ana"), (30, "bo"), (None, "unknown")])

# Reliable anti-join: LEFT JOIN + IS NULL. Keeps orphans and NULL keys.
anti = ("SELECT o.id FROM orders o LEFT JOIN customers c ON o.customer_id = c.id "
        "WHERE c.id IS NULL ORDER BY o.id")
assert con.execute(anti).fetchall() == [(2,), (3,)]

# The NOT IN trap: one NULL in the subquery makes the predicate unknown for every row,
# so the query returns nothing at all.
trap = "SELECT id FROM orders WHERE customer_id NOT IN (SELECT id FROM customers)"
assert con.execute(trap).fetchall() == []

# Filtering the NULL out repairs NOT IN, but that is easy to forget.
safe = ("SELECT id FROM orders WHERE customer_id NOT IN "
        "(SELECT id FROM customers WHERE id IS NOT NULL) ORDER BY id")
assert con.execute(safe).fetchall() == [(2,)]

# NOT EXISTS is NULL-safe without any guard clause.
exists = ("SELECT o.id FROM orders o WHERE NOT EXISTS "
          "(SELECT 1 FROM customers c WHERE c.id = o.customer_id) ORDER BY o.id")
assert con.execute(exists).fetchall() == [(2,), (3,)]
con.close()
print("NULL-safe anti-join check passed")

NULL-safe anti-join check passed


## Spark and PySpark

Spark questions test whether you reason about the shuffle, the plan and the cluster rather than just the DataFrame API. The check simulates a shuffle and lazy evaluation in plain Python.

**Q.** Explain lazy evaluation and the difference between transformations and actions.

**A.** Transformations such as `select`, `filter`, `withColumn` and `groupBy` are lazy: they build a logical plan and return a new DataFrame without touching data. Actions such as `count`, `collect`, `write` and `show` trigger the job, at which point Catalyst optimizes the plan and the DAG scheduler splits it into stages and tasks. This is why you can chain many steps cheaply, and why nothing appears until an action runs. It is also why a `print` placed inside a transformation only executes at action time, on the driver.

**Follow-up:** What is the difference between the logical plan, the optimized plan and the physical plan, and how do you inspect them?

**Q.** What is the difference between narrow and wide transformations?

**A.** Narrow transformations (`map`, `filter`, `select`) let each input partition feed exactly one output partition, so no data crosses the network. Wide transformations (`groupBy`, `join`, `distinct`, `repartition`) require a shuffle that redistributes rows by key, which is the dominant cost and the main source of skew. Shuffle boundaries also define stage boundaries, so counting stages tells you how many shuffles a job performs. Minimize shuffles, and combine or filter aggressively before them.

**Follow-up:** How does `reduceByKey` differ from `groupByKey` in RDD terms, and what is the DataFrame equivalent of that optimization?

**Q.** Broadcast join vs sort-merge join, and how do you handle skew?

**A.** A broadcast join ships the small side to every executor and avoids shuffling the large side; Spark chooses it automatically when a side is under `spark.sql.autoBroadcastJoinThreshold`, or you force it with `broadcast()`. When both sides are large, sort-merge join shuffles and sorts both by the join key. Skew happens when a few keys hold most of the rows; mitigate it by salting the hot key, splitting the skewed side, or enabling AQE skew-join handling. Confirm with the Spark UI that one straggler task, not uniformly slow tasks, is the problem.

**Follow-up:** What is a bucketed join, and how does pre-bucketing by the join key avoid the shuffle entirely?

**Q.** repartition vs coalesce?

**A.** `repartition(n)` performs a full shuffle and can increase or decrease partitions; it also accepts columns for hash partitioning. `coalesce(n)` only merges existing partitions, so it is cheap, cannot increase parallelism, and can leave uneven partition sizes. Use `coalesce` before a small write to avoid tiny files, and `repartition` when you need balanced partitions or a specific distribution before a join. Aim for roughly 100–200 MB per partition, or two to four tasks per core.

**Follow-up:** What is the small-files problem on write, and why is `coalesce(1)` usually the wrong fix?

**Q.** When do you cache or persist a DataFrame, and what does checkpointing add?

**A.** `cache()`/`persist()` materializes a DataFrame on first action, which pays off when the same lineage is reused by several actions, as in iterative ML or branching logic. Caching everything wastes memory and can evict the intermediates that actually matter, so cache only reused, expensive results. `checkpoint()` writes to reliable storage and truncates the lineage, which is essential for very long iterative lineages or streaming state. Unpersist when done, especially in long-lived sessions.

**Follow-up:** What are the storage levels (`MEMORY_AND_DISK`, `DISK_ONLY`, serialized) and when would you pick each?

**Q.** When is a UDF acceptable in PySpark, and what are the alternatives?

**A.** Native `pyspark.sql.functions` stay inside the JVM and are optimized by Catalyst, so they should be the default. A plain Python UDF forces row-by-row serialization between the JVM and Python and is opaque to Catalyst; vectorized pandas UDFs amortize that cost and are the preferred escape hatch. Use a UDF only when the logic genuinely cannot be expressed natively, such as calling a third-party library or parsing an exotic format. Measure before and after, because a UDF can turn a sub-minute job into an hour.

**Follow-up:** What is the difference between a scalar, grouped-map and map-iterator pandas UDF in memory profile and use case?

**Q.** A Spark job fails with OOM or runs with a few straggler tasks — how do you debug it?

**A.** Check the Spark UI for the failing stage, task-duration distribution, shuffle read/write sizes and input partition sizes; a handful of long tasks means skew, while uniform OOM means partitions are simply too large. Fix skew by salting or splitting hot keys, and fix partition size by repartitioning into more, smaller partitions or raising the shuffle partition count. Reduce data early with filters and column pruning, and never collect large results to the driver. Raising executor memory without fixing skew usually just delays the failure.

**Follow-up:** What does Adaptive Query Execution change at runtime, and which three problems does it address by default?

### Verify your answer: map/shuffle/reduce and lazy evaluation

In [9]:
import itertools
from collections import defaultdict

def flat_map(pairs):
    for key, value in pairs:
        for token in value.lower().split():
            yield (token, 1)

def shuffle(pairs):
    return sorted(pairs, key=lambda kv: kv[0])       # the network exchange

def combine_local(pairs):
    local = defaultdict(int)
    for k, v in pairs:
        local[k] += v
    return list(local.items())                       # map-side partial aggregation

def reduce_by_key(pairs):
    out = {}
    for key, group in itertools.groupby(pairs, key=lambda kv: kv[0]):
        out[key] = sum(v for _, v in group)
    return out

data = [("a", "spark sql"), ("b", "SPARK streaming"), ("c", "sql")]
counts = reduce_by_key(shuffle(combine_local(list(flat_map(data)))))
assert counts == {"spark": 2, "sql": 2, "streaming": 1}

# Lazy evaluation: a transformation builds a plan; only an action runs it.
calls = []

def transform(record):
    calls.append(record)
    return record.upper()

plan = map(transform, ["a", "b"])
assert calls == []                                   # nothing executed yet
assert list(plan) == ["A", "B"] and calls == ["a", "b"]
print("map/shuffle/reduce check passed")

map/shuffle/reduce check passed


## Data modelling and warehousing

Modelling questions probe whether you can pick a grain, preserve history and keep BI fast. The check builds a small SCD Type 2 dimension in memory.

**Q.** Star schema vs snowflake schema — which do you choose?

**A.** A star schema keeps denormalized dimensions joined directly to a central fact table, which gives simple, fast queries and fewer joins; a snowflake normalizes dimensions into sub-tables, saving storage and easing maintenance of shared hierarchies. In columnar warehouses storage is cheap and joins are the expensive part, so stars usually win, especially for BI. Snowflake can still help with very large, slowly changing dimensions or shared conformed hierarchies. Whatever you choose, declare and enforce the grain of the fact table — what one row represents.

**Follow-up:** What is a conformed dimension, and why does the grain of a fact table drive every downstream metric?

**Q.** Walk through SCD Type 1, 2 and 3.

**A.** Type 1 overwrites the attribute: history is lost but the dimension stays small. Type 2 adds a new row per change with `valid_from`, `valid_to` and a current flag, preserving full history at the cost of a growing table. Type 3 keeps a limited number of prior values in extra columns and is rarely worth it. Type 2 is the standard for auditable dimensions and is what makes point-in-time as-of joins possible. Implement it with a merge on the business key plus a change-detection hash.

**Follow-up:** How do you handle a Type 2 dimension when a source row is deleted, or re-appears after being inactive?

**Q.** When do you normalize and when do you denormalize in a warehouse?

**A.** Normalize in operational systems where writes dominate and consistency matters, because it removes redundancy and update anomalies. Denormalize in analytics where reads dominate and joins over billions of rows dominate cost, so wide stars and materialized views are usually faster than repeated joins. Normalization still belongs in the staging layer, where conformance and dedupe are the goals. Choose by workload characteristics rather than by dogma.

**Follow-up:** What anomalies does 3NF prevent, and why is a fully normalized model often slow in a columnar engine?

**Q.** What is a factless fact table, and what is an accumulating snapshot?

**A.** A factless fact table records an event or relationship with no measures — attendance, eligibility, coverage — and is queried with `COUNT(*)`. An accumulating snapshot keeps one row per process instance and updates it as the instance hits milestones such as placed, shipped and delivered, with one date key per milestone, which makes cycle-time analysis easy. Transaction facts and periodic snapshot facts are the other two core types. Pick the fact type by the question it must answer, not by the shape of the source table.

**Follow-up:** How would you model order-lifecycle metrics so that late-arriving milestone events are handled correctly?

**Q.** How do you handle late-arriving facts and dimensions?

**A.** For late facts, partition by event date and reprocess the affected partitions from the source of truth, or upsert on a natural key, instead of appending blindly. For late dimensions, either insert a placeholder row that is updated later (an inferred member), or use late-binding views so the fact-to-dimension join resolves at query time. Type 2 dimensions plus as-of joins answer 'what did it look like then' precisely. Whichever approach you take, define a lateness SLA so the pipeline knows when it may stop waiting.

**Follow-up:** What is an inferred member, and what breaks in a Type 2 dimension when a fact arrives before its dimension row?

**Q.** Kimball vs Inmon vs Data Vault — how do they differ?

**A.** Kimball builds a dimensional bus of conformed stars, optimized for business-facing analytics and fast delivery. Inmon builds a normalized enterprise warehouse as the single source of truth, with marts downstream, favouring integration and auditability over query simplicity. Data Vault models hubs, links and satellites to absorb change and preserve audit history, with business vaults and marts on top; it scales well in regulated, fast-changing source landscapes but adds modelling overhead. Many modern stacks are pragmatic hybrids: raw and refined layers, then dimensional marts.

**Follow-up:** When would Data Vault's insert-only design genuinely pay off, and what does it cost you?

**Q.** How do partitioning, clustering and sort keys affect warehouse performance and cost?

**A.** Partitioning splits storage by a column, usually a date, so predicates prune whole partitions and reduce bytes scanned, which is the dominant cost driver in BigQuery and Snowflake. Clustering and sort keys order data within partitions so range and equality filters skip blocks and joins gain locality. Choose high-selectivity, frequently filtered columns with moderate cardinality for partitions, because over-partitioning creates small files and metadata overhead. Verify with the plan or the bytes-scanned metric that pruning is actually happening.

**Follow-up:** Why is partitioning by a high-cardinality ID usually a mistake, and how does Snowflake's automatic clustering differ from static partitioning?

### Verify your answer: SCD Type 2 history and as-of joins

In [10]:
def scd2(history, updates, day):
    for key, new_value in updates.items():
        rows = history[key]
        current = rows[-1]
        if current["value"] != new_value:
            current["valid_to"] = day
            current["is_current"] = False
            rows.append({"value": new_value, "valid_from": day,
                         "valid_to": None, "is_current": True})
    return history

h = {"c1": [{"value": "gold", "valid_from": "2024-01-01",
             "valid_to": None, "is_current": True}]}
scd2(h, {"c1": "gold"}, "2024-02-01")                 # no change -> no new version
assert len(h["c1"]) == 1
scd2(h, {"c1": "platinum"}, "2024-03-01")
assert h["c1"][0]["valid_to"] == "2024-03-01" and h["c1"][0]["is_current"] is False
assert h["c1"][1]["value"] == "platinum" and h["c1"][1]["is_current"] is True

def as_of(rows, day):
    return [r for r in rows
            if r["valid_from"] <= day and (r["valid_to"] is None or day < r["valid_to"])]

assert [r["value"] for r in as_of(h["c1"], "2024-02-15")] == ["gold"]
assert [r["value"] for r in as_of(h["c1"], "2024-03-15")] == ["platinum"]
print("SCD type 2 check passed")

SCD type 2 check passed


## Lakehouse and file formats

These questions test the physical layer: formats, table standards and the small-files economics. The check models Delta/Iceberg snapshots and time travel.

**Q.** Compare Parquet, ORC, Avro and JSON.

**A.** Parquet and ORC are columnar, splittable and compressed, ideal for analytics where you scan a few columns of wide tables; ORC is common in Hive, Parquet in Spark and the broader lake. Avro is row-oriented with a schema and is good for write-heavy ingestion, Kafka payloads and records read end to end. JSON is text, self-describing and human-readable, but it is the slowest and most verbose at scale — fine at the edge, expensive in the core. Columnar formats embed statistics that enable predicate pushdown and column pruning.

**Follow-up:** How do Parquet row groups, pages and column statistics enable predicate pushdown, and why is a nested JSON blob hostile to it?

**Q.** How do Delta Lake, Iceberg and Hudi bring ACID to object storage?

**A.** All three add a metadata log over immutable data files: a write creates new files and atomically commits a new table version, so readers never observe a partial write. Delta uses a JSON transaction log plus periodic checkpoints; Iceberg uses manifest lists and manifests tied to snapshots; Hudi uses a timeline with copy-on-write or merge-on-read tables. That metadata also delivers time travel, schema evolution and optimistic concurrency. The trade-offs are write amplification for copy-on-write versus read amplification for merge-on-read, plus catalog and ecosystem maturity.

**Follow-up:** What is the difference between copy-on-write and merge-on-read, and which would you choose for heavy upserts?

**Q.** What is the small-files problem, and how do you fix it?

**A.** Many tiny files mean one task per file, so scheduling and metadata overhead dominate and the job spends its time opening files rather than reading data. It is caused by streaming micro-batches, over-partitioning, or a high shuffle-partition count on write. Fix it with compaction (`OPTIMIZE`/`rewrite_data_files`), `coalesce` or `repartition` before writes, and by targeting files of roughly 128–512 MB. Table formats can automate compaction, but you still have to tune partition granularity or it comes back.

**Follow-up:** Why can compaction hurt concurrent readers and writers, and how do Iceberg manifest rewrites differ from Delta OPTIMIZE?

**Q.** How do time travel and schema evolution work in a lakehouse table?

**A.** Every commit creates an immutable snapshot, and time travel resolves an older snapshot by ID or timestamp, which supports auditing, reproducible backfills and rollback. Schema evolution adds, renames or widens columns by tracking field IDs in metadata, so old files do not need rewriting; readers reconcile files written under different schemas. Safe changes are additive, while narrowing types or dropping columns used downstream needs coordination. Iceberg's column mapping and Delta's column-mapping mode make renames reliable.

**Follow-up:** What is the difference between schema evolution and schema enforcement, and what happens on a type change such as INT to STRING?

**Q.** Describe the medallion (bronze/silver/gold) architecture.

**A.** Bronze holds raw ingested data as close to the source as possible, append-only with ingestion metadata. Silver cleans, deduplicates, conforms types and applies business rules into reusable entity-level tables. Gold holds aggregated, business-facing marts tuned for specific consumption patterns. The layers separate concerns so a reprocess can restart at the right level, and each layer is a contract for the next. Keep the transformations declarative and idempotent so any layer can be rebuilt from the one below.

**Follow-up:** Where do you put data-quality quarantine rows, and how do you version the contracts between layers?

**Q.** How do you choose partitioning and file sizes for a lake table?

**A.** Partition on low-to-moderate cardinality columns that appear in almost every filter, typically event date, and optionally a coarse region or tenant. Target files of roughly 128–512 MB and avoid partitions much smaller than that, or you recreate the small-files problem. Within partitions, sort or Z-order by frequent join keys and high-cardinality filters to improve data skipping. Iceberg's hidden partitioning lets you partition on a derived value without exposing the transform to every query.

**Follow-up:** What does Z-ordering optimize compared with plain sorting, and what is its main cost?

**Q.** Why do you need a catalog with a lakehouse table format?

**A.** The catalog maps a table name to the current metadata pointer and performs atomic pointer swaps, which is how concurrent writers commit safely; it also centralizes access control and discovery. Without a catalog, listing object storage is slow, non-atomic and cannot give snapshot isolation. Common options are Hive Metastore, AWS Glue, Nessie, Unity Catalog and REST catalogs. A catalog is also what makes multi-engine interop practical: Spark writes while Trino and Snowflake read the same table.

**Follow-up:** What problem does a REST catalog or Nessie solve for multi-engine and branching workflows?

### Verify your answer: immutable snapshots and time travel

In [11]:
class Table:
    """Tiny Iceberg/Delta stand-in: immutable snapshots over data files."""
    def __init__(self):
        self.files = []                              # list of (path, records)
        self.snapshots = []                          # list of {"id", "files"}
        self._commit()
    def _commit(self):
        self.snapshots.append({"id": len(self.snapshots), "files": tuple(self.files)})
    def append(self, path, records):
        self.files.append((path, list(records)))
        self._commit()
    def read(self, snapshot=None):
        snap = self.snapshots[-1] if snapshot is None else self.snapshots[snapshot]
        rows = []
        for _path, records in snap["files"]:
            rows.extend(records)
        return rows

t = Table()
assert t.read(snapshot=0) == []                      # empty initial snapshot
t.append("f1.parquet", [{"id": 1}, {"id": 2}])
t.append("f2.parquet", [{"id": 3}])
assert t.read() == [{"id": 1}, {"id": 2}, {"id": 3}]
assert t.read(snapshot=1) == [{"id": 1}, {"id": 2}]  # time travel before f2
assert len(t.snapshots) == 3 and t.snapshots[-1]["id"] == 2

# Schema evolution: a later file adds a column, old files still read.
t.append("f3.parquet", [{"id": 4, "country": "US"}])
rows = t.read()
assert rows[-1] == {"id": 4, "country": "US"}
assert "country" not in rows[0]                      # old rows keep the old shape
print("lakehouse time-travel check passed")

lakehouse time-travel check passed


## Pipelines and orchestration

Orchestration questions test reliability engineering: idempotency, retries, backfills and testing. The two checks below exercise retry/backoff and idempotent upserts.

**Q.** What does idempotency mean for a data pipeline, and how do you get it?

**A.** A pipeline is idempotent when re-running it with the same input produces the same target state instead of duplicates. Achieve it with deterministic keyed upserts (`MERGE` on a business key), partition overwrite instead of append, and dedupe windows on a natural key plus event time. Add a run or commit marker so a partially written batch is never double-counted. Idempotency is what makes retries and backfills safe to run without human inspection.

**Follow-up:** How do you make an exactly-once sink when the downstream system has no transactions?

**Q.** How do you choose between Airflow, Dagster, dbt and a plain scheduler?

**A.** dbt is a SQL transformation framework with lineage, tests and incremental materializations; it complements an orchestrator rather than replacing it. Airflow is the mature, general-purpose DAG scheduler with the widest provider ecosystem and is usually the safe enterprise default. Dagster is asset-oriented, with typed I/O, software-defined assets and first-class local testing, which suits data-centric teams. Choose on team skills, backfill needs and whether assets or tasks are the natural unit of work.

**Follow-up:** How do sensors, deferrable operators and event-driven triggers change the cost and latency of an Airflow DAG?

**Q.** How do you run a safe backfill?

**A.** Reproduce the partition logic deterministically, then write into isolated target partitions or a shadow table and verify before swapping into production. Bound concurrency so the backfill neither starves the daily schedule nor blows up warehouse cost, and keep it idempotent so it can resume. Compare row counts and key metrics against the previous version before cutover. Record the exact parameters so the run is reproducible and auditable.

**Follow-up:** What makes a backfill non-deterministic, and how do you handle late-arriving data while reprocessing?

**Q.** How do you design retries, backoff and dead-letter handling?

**A.** Retry only transient failures such as network errors, throttling and deadlocks, and fail fast on permanent ones such as schema mismatches or bad credentials. Use exponential backoff with jitter to avoid synchronized retry storms, and cap both attempts and total elapsed time. Make the task idempotent so a retry cannot double-write. After the budget is exhausted, route the payload to a dead-letter location with error context for replay, and alert on exhaustion rather than on every individual retry.

**Follow-up:** How do you distinguish transient from permanent failures in an API integration, and when do you park data instead of failing the DAG?

**Q.** What data-quality checks belong in a pipeline?

**A.** Start with freshness, volume, schema and key-level uniqueness and nullability, plus referential integrity between facts and dimensions. Add business rules such as allowed ranges, accepted enum values and reconciliation totals against the source. Decide per check whether to fail the pipeline or quarantine bad rows, based on the cost of being wrong, and make that choice explicit. Publish the results as metrics so drift is visible before a downstream consumer complains.

**Follow-up:** How do you decide between failing the pipeline, warning, and quarantining bad rows?

**Q.** How do you do CI/CD for data pipelines?

**A.** Version everything — code, SQL models, infrastructure and job definitions — and run tests on every change: unit tests on transformations, data tests against a seed or sampled dataset, and SQL linters. Use isolated dev and staging schemas plus a `--check` mode that executes without writing to production. Promote artefacts by reference with a blue-green or canary deployment, and keep schema migrations backward compatible. Automate rollback by retaining previous artefacts and table snapshots.

**Follow-up:** How do you test a SQL transformation without copying production data, and what role does a seed dataset play?

**Q.** Time-driven or event-driven scheduling — how do you choose?

**A.** Time-driven schedules are simple, predictable and easy to backfill, but they add latency and may run on incomplete data. Event-driven triggers — file arrival, Kafka message, table-update sensor — reduce latency and wasted runs, but demand idempotent consumers and dead-letter handling for duplicates and out-of-order events. A common hybrid is event-driven ingestion with time-driven aggregation. Whichever you choose, decouple the schedule from the data-completeness contract.

**Follow-up:** How do you detect that a batch is complete when the source only emits files as they arrive?

### Verify your answer: retry with exponential backoff and fail-fast

In [12]:
def retry_call(fn, attempts=4, base=0.5, sleep=lambda seconds: None):
    delays = []
    for attempt in range(attempts):
        try:
            return fn(), delays
        except RuntimeError:
            if attempt == attempts - 1:
                raise
            delay = base * (2 ** attempt)            # exponential backoff
            delays.append(delay)
            sleep(delay)

state = {"calls": 0}

def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise RuntimeError("transient")
    return "ok"

result, delays = retry_call(flaky)
assert result == "ok" and delays == [0.5, 1.0] and state["calls"] == 3

# A permanent error is not retried: fail fast.
def permanent():
    raise ValueError("bad schema")

try:
    retry_call(permanent)
    raise AssertionError("permanent error must not be retried")
except ValueError:
    pass

# Budget exhausted -> the last error propagates.
seen = []

def always_failing():
    seen.append(len(seen))
    raise RuntimeError("still down")

try:
    retry_call(always_failing, attempts=3)
    raise AssertionError("should have raised")
except RuntimeError as exc:
    assert str(exc) == "still down" and seen == [0, 1, 2]
print("retry/backoff check passed")

retry/backoff check passed


### Verify your answer: idempotent upsert vs duplicate-producing append

In [13]:
# Keyed upsert: re-running the same batch must not change the target state.
def merge(table, batch, key="id"):
    out = {row[key]: dict(row) for row in table}
    for row in batch:
        out[row[key]] = dict(row)
    return sorted(out.values(), key=lambda row: row[key])

target = [{"id": 1, "amount": 10}]
batch = [{"id": 1, "amount": 99}, {"id": 2, "amount": 20}]
once = merge(target, batch)
twice = merge(once, batch)
assert once == twice == [{"id": 1, "amount": 99}, {"id": 2, "amount": 20}]

# Partition overwrite is idempotent as well: replace, never append.
def overwrite_partition(partitions, partition, rows):
    updated = {key: list(value) for key, value in partitions.items()}
    updated[partition] = list(rows)
    return updated

p = overwrite_partition({"2024-01-01": [1, 2]}, "2024-01-01", [1, 2, 3])
assert p == {"2024-01-01": [1, 2, 3]}
assert overwrite_partition(p, "2024-01-01", [1, 2, 3]) == p
assert "2024-01-02" not in p                         # untouched partitions survive

# An append is NOT idempotent: the same batch counted twice.
def append(target_rows, batch_rows):
    return list(target_rows) + list(batch_rows)

assert append([1], [2]) != append(append([1], [2]), [2])
print("idempotent upsert check passed")

idempotent upsert check passed


## Streaming

Streaming interviews centre on delivery guarantees, time semantics and state. The check models an event-time window with a watermark and late-data policy.

**Q.** How do Kafka partitions, offsets and consumer groups work together?

**A.** A topic is split into partitions, each an ordered append-only log, so ordering is guaranteed only within a partition, never across the topic. A consumer group distributes partitions among its members so each partition is read by exactly one member, giving parallelism up to the partition count. Offsets are committed per partition to track progress, and consumers can reset them to replay history. More partitions buy parallelism but add metadata, rebalance time and end-to-end latency.

**Follow-up:** What happens to partition assignment during a rebalance, and how does cooperative/incremental rebalancing reduce the pause?

**Q.** At-least-once vs at-most-once vs exactly-once?

**A.** At-most-once may lose messages but never duplicates; at-least-once never loses but may duplicate, so consumers must be idempotent. Exactly-once requires coordination — Kafka transactions with idempotent producers and transactional sinks, or dedupe on a deterministic key downstream. In practice 'effectively-once' (at-least-once delivery plus an idempotent sink keyed on event ID) is the pragmatic standard. State the guarantee end to end, because every hop in the chain can weaken it.

**Follow-up:** How does Flink checkpointing give exactly-once state, and what happens with sinks that cannot join a two-phase commit?

**Q.** Event time vs processing time, and what is a watermark?

**A.** Event time is when the event occurred; processing time is when the system saw it, and the gap is caused by network delay, retries and out-of-order arrival. A watermark is the system's estimate that no events older than it will arrive, which lets event-time windows close and emit results. Setting it too aggressively drops late data; too laxly holds windows and state open, increasing latency and memory. Late events beyond the watermark go to a side output or are dropped according to policy.

**Follow-up:** How do allowed lateness and a side output for dropped events change the accuracy/latency trade-off?

**Q.** Compare tumbling, sliding and session windows.

**A.** Tumbling windows are fixed-size and non-overlapping, which is the simplest and cheapest option. Sliding windows are fixed-size and overlap by a configured step, so each event can belong to several windows — more state and compute, but smoother metrics. Session windows are dynamic and close after a gap of inactivity, which suits user-activity analysis. In all cases, event-time semantics plus watermarks decide when a window emits. Choose the window from the metric: exact periodic totals, smoothed trends, or user sessions.

**Follow-up:** How does the state requirement of a sliding window grow with the slide size, and how do you bound it?

**Q.** Kafka vs Kinesis vs Pulsar — what actually differs?

**A.** Kafka is the ecosystem default, self-managed or managed, with a huge connector ecosystem and partition-based ordering. Kinesis is AWS-native and serverless with shards and configurable retention up to a year; it is simpler to operate but has tighter payload limits and AWS coupling. Pulsar separates compute (brokers) from storage (BookKeeper), supports multi-tenancy and tiered storage, and offers both queueing and streaming semantics. Decide on operational capacity, cloud strategy, ordering and retention needs, and existing tooling.

**Follow-up:** How does Pulsar's subscription model (exclusive, shared, failover, key-shared) change ordering guarantees?

**Q.** How does CDC work, and log-based vs query-based?

**A.** Log-based CDC tails the database write-ahead log with a tool such as Debezium, emitting row-level inserts, updates and deletes with minimal source impact and near-real-time latency. Query-based CDC polls with an `updated_at` filter or a merge, which is simpler but can miss hard deletes, adds source load, and depends on a reliable change timestamp. Log-based output includes before/after images and an operation type, which downstream can replay into an idempotent merge. Both approaches must handle schema changes and an initial full snapshot explicitly.

**Follow-up:** How do you bootstrap a full snapshot and then transition to streaming without missing or duplicating changes?

**Q.** How do you handle consumer lag and backpressure?

**A.** Measure first: lag per partition over time, consumer processing time, and rebalance rate. Scale consumers up to the partition count and add partitions if that is the bottleneck; then optimize the consumer with batching, async I/O and fewer per-message remote calls. Apply backpressure or bounded queues so a slow sink cannot cause unbounded memory growth and crashes. For persistent lag, decouple with a buffer, shed load, or increase parallelism at the sink.

**Follow-up:** Why can adding consumers fail to reduce lag, and what does that tell you about the bottleneck?

### Verify your answer: event-time windows, watermark and late data

In [14]:
WINDOW = 10          # seconds, tumbling
WATERMARK = 5        # seconds of allowed lateness

def run(events, max_event_time=None):
    if max_event_time is None:
        max_event_time = max(ts for ts, _key in events)
    horizon = max_event_time - WATERMARK
    counts, dropped = {}, []
    for ts, key in events:
        if ts < horizon:
            dropped.append((ts, key))                # behind the watermark
            continue
        window = (ts // WINDOW) * WINDOW
        counts[(window, key)] = counts.get((window, key), 0) + 1
    return counts, dropped

# Latest event time is 14, so the watermark sits at 14 - 5 = 9.
# The event at t=2 arrives after that horizon and is therefore dropped.
events = [(9, "a"), (11, "a"), (12, "b"), (14, "b"), (2, "a")]
counts, dropped = run(events)
assert counts == {(0, "a"): 1, (10, "a"): 1, (10, "b"): 2}
assert dropped == [(2, "a")]

# Same events with a slower clock: the watermark has not passed t=2, so it is counted.
counts2, dropped2 = run([(0, "a"), (1, "b"), (2, "a")], max_event_time=4)
assert dropped2 == [] and counts2 == {(0, "a"): 2, (0, "b"): 1}
print("streaming watermark check passed")

streaming watermark check passed


## Cloud, cost and reliability

Cloud rounds probe cost intuition and operational maturity. The check quantifies columnar scan savings and a simple per-query cost model.

**Q.** How do columnar storage and compression reduce cloud cost?

**A.** Columnar formats let a query read only the referenced columns, and compression with encoding such as dictionary, run-length and delta shrinks bytes scanned, which is the billing unit in BigQuery and Snowflake and the main driver of S3 scan time. Row-oriented CSV or JSON forces reading whole rows and usually stores far more bytes, which is why converting landing data to Parquet early pays for itself. Compression also lowers storage and egress cost, at the price of CPU on write. Sorting and clustering the data lets the engine skip even more.

**Follow-up:** Why can a heavily compressed file be slower to read for a highly selective point lookup?

**Q.** How do spot/preemptible instances and autoscaling affect reliability?

**A.** Spot capacity is cheap but reclaimable, so it suits stateless, retryable work such as batch Spark executors, stateless consumers and CI — not stateful singletons like databases or the Spark driver. Keep checkpointing and shuffle spill on durable storage so a lost executor's work can be recomputed, and treat reclamation as an expected event rather than an incident. Drive autoscaling from backlog or scan size, not just CPU, and keep a warm pool to avoid cold-start latency. The saving is real but must be priced against the retry cost.

**Follow-up:** What state must live on durable storage for a Spark job to survive losing its driver, and what does dynamic allocation change?

**Q.** How do partitioning and pruning affect the bill in BigQuery or Snowflake?

**A.** These engines bill by bytes scanned or compute time, so pruning partitions and clusters is the single biggest lever: a date filter on a partitioned table can cut a query from terabytes to gigabytes. Select only needed columns, avoid `SELECT *`, and choose clustering keys that match the most common filters. Materialized views and pre-aggregated marts turn repeated expensive scans into cheap reads. Monitor bytes-scanned per query and set quotas or custom cost controls to catch runaway patterns.

**Follow-up:** How does Snowflake warehouse suspend/resume interact with clustering cost, and when does clustering stop paying for itself?

**Q.** What do you monitor for data reliability, and what SLOs would you set?

**A.** Monitor pipeline success and duration, freshness (time since the last good partition), volume against a seasonal baseline, schema changes, and data-quality test results. For streaming, add consumer lag, checkpoint duration and watermark delay. Set SLOs on freshness and completeness per critical dataset, with error budgets, and alert on symptom-level breaches rather than every task failure. Tie issues that affect downstream consumers to an explicit on-call path.

**Follow-up:** How do you choose a freshness SLO, and how do you avoid alert fatigue from noisy volume checks?

**Q.** How do you plan for disaster recovery — RPO and RTO?

**A.** RPO is how much data you can afford to lose and RTO is how long until service is restored; both are business decisions that determine architecture and budget. A baseline plan is infrastructure as code plus replicated object storage and catalog metadata so the lakehouse can be rebuilt in another region; a stronger one adds cross-region replication and warm standby compute. Test restores regularly, because an untested backup is a hope rather than a plan. Document the failover runbook and who is authorized to execute it.

**Follow-up:** What must be replicated for a lakehouse table to be usable in another region: data files, catalog metadata, or both?

**Q.** What are the core security controls for a data platform?

**A.** Use least-privilege IAM with role-based access, avoid long-lived keys where workload identity or OIDC is possible, and scope service accounts per job. Encrypt in transit and at rest with managed keys, and keep secrets in a secrets manager rather than in code or notebooks. Classify data, mask or tokenize PII at ingestion, and enforce row- and column-level policies in the warehouse and catalog. Log and audit access to sensitive tables, and maintain a deletion and retention process for privacy requests.

**Follow-up:** How do column-level masking, row-level security and dynamic views differ in where the policy is enforced?

**Q.** How do you govern and reduce cloud data cost?

**A.** Make cost visible per team and per dataset with tags and chargeback, then attack the big levers: bytes scanned (partitioning, clustering, removing `SELECT *`), idle warehouses and clusters (auto-suspend and right-sizing), storage lifecycle tiers and deletion of unreferenced files, and duplicate computation (caching, materialized marts). Set budgets and anomaly alerts so a runaway query or job is caught the same day. Right-size on weeks of utilization, not on peaks. Cheaper is not free if it costs a page at 3 a.m.

**Follow-up:** How would you find the top three cost drivers in a warehouse you have never seen before?

### Verify your answer: columnar scan savings and per-query cost

In [15]:
# Columnar scan of one column vs a row-oriented full scan of a wide table.
COLUMNS = {"id": 8, "ts": 8, "user": 16, "payload": 200}   # bytes per row
ROWS = 1_000_000
COMPRESSION = 4                                            # typical Parquet+Snappy

row_bytes = sum(COLUMNS.values()) * ROWS
col_bytes = COLUMNS["user"] * ROWS
assert row_bytes == 232_000_000
assert col_bytes == 16_000_000

scan_savings = row_bytes / (col_bytes / COMPRESSION)
assert scan_savings == 58.0

# Cost model: $5 per TB scanned, 30 such queries per day.
TB = 1024 ** 4
daily_bytes = col_bytes / COMPRESSION * 30
daily_cost = daily_bytes / TB * 5
assert daily_cost < 0.01
assert daily_cost == (col_bytes / COMPRESSION * 30) / TB * 5

# Sorting/clustering the column prunes even more: assume 10x fewer bytes read.
clustered_cost = daily_cost / 10
assert clustered_cost < daily_cost
print("cost/columnar check passed:", round(scan_savings, 1), "x less data scanned")

cost/columnar check passed: 58.0 x less data scanned


## Behavioural and system design

These close the interview: structure, trade-offs and communication. There is no runnable check here — rehearse them out loud against a timer.

**Q.** How do you structure a behavioural answer?

**A.** Use STAR — Situation, Task, Action, Result — but spend most of the time on your actions and decisions rather than the backstory. Quantify the result (latency cut, cost saved, incident avoided) and be explicit about the trade-off you chose and why. Prepare five to seven stories that can be reframed across questions such as conflict, failure, leadership, ambiguity and delivery under pressure. Keep the first version to about two minutes, then let the interviewer dig.

**Follow-up:** How do you answer 'tell me about a failure' without sounding either defensive or self-destructive?

**Q.** Design a batch pipeline that ingests e-commerce orders and serves daily analytics.

**A.** Land raw order events from the source via CDC or files into bronze in Parquet/Avro with ingestion metadata, then clean, dedupe by order ID and conform types into silver order and order-line entities. Model gold as a star: an order fact at the declared grain plus customer, product and date dimensions, with SCD Type 2 on the dimensions. Orchestrate with an idempotent, partition-overwriting DAG that supports date-based backfills and runs freshness and volume checks between layers. Partition by event date, target 128–512 MB files, and expose the gold marts to BI. Handle late data with a lateness SLA and targeted partition reprocessing.

**Follow-up:** Where exactly do you deduplicate and why, and what happens to an order that is updated after the daily cut?

**Q.** Design real-time clickstream analytics for a product dashboard.

**A.** Produce click events with a stable event ID and event timestamp to Kafka, keyed by user where ordering matters, with enough partitions for peak throughput. Stream-process with Flink or Spark Structured Streaming using event-time windows and watermarks, writing aggregates to a serving store such as ClickHouse, Druid or Bigtable, and raw data to the lake for replay and correction. The sink must be idempotent or transactional, keyed on event ID, to survive at-least-once delivery. Serve live or approximate metrics on the dashboard and batch-corrected totals for finance-grade reporting. Define the lateness policy and monitor lag and watermark delay.

**Follow-up:** How would you make the streaming and batch views agree (lambda vs kappa), and which would you choose here?

**Q.** How do you handle ambiguity or changing requirements?

**A.** Restate the goal and the decision you need from the stakeholder, ask what 'done' and the deadline mean, and write your assumptions down explicitly. Prototype the smallest end-to-end slice to force concrete feedback early instead of debating abstractly. Present options with trade-offs and a recommendation rather than an open-ended question, and confirm the choice in writing. If requirements still move, keep the architecture reversible and the data model extensible.

**Follow-up:** Describe a time you pushed back on a requirement — what was the outcome?

**Q.** A pipeline produced wrong numbers in an executive dashboard. How do you handle it?

**A.** Stop the bleeding first: pause the downstream refresh or flag the dashboard, and quantify how wrong and which dates are affected. Communicate early with impact and an ETA, then trace from the faulty metric back through the lineage to the first bad transformation. Fix it, backfill the affected partitions idempotently, and verify against an independent source before restoring. Finish with a blameless postmortem, a regression test for that exact failure, and a monitoring change so the same class of bug is caught earlier next time.

**Follow-up:** What test would you have added so this specific failure could not recur silently?

**Q.** How do you prioritize when you have more requests than capacity?

**A.** Make the trade-off explicit with a shared framework — impact on revenue or risk, effort, and reversibility — and get the priority confirmed by whoever owns the outcome. Reserve a fixed slice of capacity for reliability and technical debt, because it never wins a head-to-head comparison otherwise. Say no to the work, not to the person: offer a smaller version, a later date, or a workaround. Revisit priorities on a predictable cadence so the backlog does not silently rot.

**Follow-up:** How do you decide to pay down technical debt when its cost is invisible until it fails?

**Q.** How do you do a back-of-the-envelope capacity estimate for a design?

**A.** Establish event rate, average record size, peak-to-average ratio and retention window, then multiply to get daily and total volume. Convert to storage with a compression factor and a replication factor, and to throughput by dividing by the SLA window to size partitions and cluster cores. Estimate cost from bytes processed or node-hours, and compare alternatives at the same scale. Sanity-check the answer against a similar known system, and state assumptions out loud so the interviewer can correct them.

**Follow-up:** For 1 billion 500-byte events per day with 90-day retention and 4x compression, what is the rough storage footprint?

## How to drill this notebook

A 45-minute solo mock interview that uses this bank plus the other notebooks in
`notebooks/`:

| Minutes | Drill | Source |
| --- | --- | --- |
| 0–5 | Warm up on language mechanics: mutable defaults, closures, generators | `notebooks/01_python_basics.ipynb` |
| 5–15 | Two timed coding problems, one easy and one medium | `notebooks/02_python_coding_problems.ipynb` |
| 15–30 | Three SQL problems: top-N per group, a NULL-safe anti-join, an incremental upsert | `notebooks/03_sql_interview_questions.ipynb` |
| 30–40 | Two PySpark questions: a join choice and a skew/OOM diagnosis | `notebooks/04_pyspark_concepts.ipynb` |
| 40–45 | One system-design prompt from this notebook (batch or streaming), answered out loud | this notebook, Behavioural and system design |

Suggested weekly rotation: pick **one section per day** from this bank, answer all
of its questions verbally, then run that section's verification cells last so the
code confirms what you just said. Once a week, take the full 45-minute run above
and only look at the model answers afterwards. Keep a running list of the
follow-ups you fumbled — those are your revision list, not the questions you
already know.